# Elias Modeling Info

This notebook combines the key reference material from `cheat_sheet.ipynb` and `report.ipynb` in one coherent, non-redundant structure.

Use `workflow.ipynb` for execution. Use this notebook for conceptual orientation, variable definitions, and modeling rationale.

## 1. Scope and Boundaries

| Area | In Scope | Out of Scope |
|---|---|---|
| Data source | Real participant data from `data/participants.csv` | Surrogate or synthetic dataset generation |
| Simulation backend | Use `src/evan/glaze.py` unchanged | Modifying Evan's backend dynamics |
| Workflow design | Simplified train/test model comparison in `workflow.ipynb` | Legacy Step3/Step4/Step5 artifact pipeline |

In [ ]:
from pathlib import Path
import pandas as pd

_candidate_csv_paths = [
    Path('data/participants.csv'),
    Path('../../data/participants.csv'),
    Path('../data/participants.csv'),
]
DATA_CSV_PATH = next((p for p in _candidate_csv_paths if p.exists()), None)
if DATA_CSV_PATH is None:
    raise FileNotFoundError('Could not find participants.csv in expected locations.')

participants_df = pd.read_csv(DATA_CSV_PATH)
print(f'Loaded {len(participants_df)} rows from {DATA_CSV_PATH}')
display(participants_df.head())


## 2. Hazard-Rate Variables and Recommended Use

### Definitions
| Variable | Where it comes from | Granularity | Role in current workflow |
|---|---|---|---|
| `hazard_rate` | Raw export column in `data/participants.csv` | Trial-level | Metadata/baseline only (not active `H`) |
| `subjective_h_snapshot` | Legacy app/export running estimate | Trial-level | Audit/sensitivity only (not active `H`) |
| `fitted_subjective_h` | Re-estimated in code via `fit_blockwise_subjective_h_choice_only(...)` on TRAIN choices | Participant-block level | Active latent hazard used as `H` for belief-state reconstruction |

### Practical tradeoffs when choosing `H` for fitting
| Candidate `H` source | Pros | Cons |
|---|---|---|
| `hazard_rate` | Deterministic, simple baseline. | Can miss participant-specific subjective volatility. |
| `subjective_h_snapshot` | High trialwise flexibility. | Leakage risk and instability for predictive evaluation. |
| `fitted_subjective_h` | Train/test clean, stable, participant-block specific. | Coarser than trialwise dynamics (one value per block). |

### Recommendation
- Use `fitted_subjective_h` as the default active `H` in model comparison.
- Use `hazard_rate` as a normative control baseline.
- Use `subjective_h_snapshot` only for explicit sensitivity analyses with leakage checks.

## 3. Workflow Architecture and Rationale

### Code structure (`src/elias/elias_models`)
- `core_workflow.py`: orchestration (`prepare_modeling_data -> fit_models_train_split -> score_models_test_split`).
- `data_pipeline.py`: loading, preprocessing integration, subjective-hazard fitting, normative state reconstruction.
- `model_fitting.py`: bounded parameterization + multi-start local search.
- `model_scoring.py`: simulation-based likelihood scoring (choice-only, RT-only conditional, joint).
- `test_core_workflow.py`: regression checks for required outputs and ranking consistency.

### End-to-end logic
1. **Load + preprocess**: normalize coding, apply exclusions, assign TRAIN/TEST.
2. **Infer `H` + rebuild belief state**: fit `H` on TRAIN choices per participant-block; reconstruct `prev_normative_belief_L`, `psi_t`, `normative_belief_L`.
3. **Fit candidate models on TRAIN**: estimate model parameters under bounded search.
4. **Score on held-out TEST**: evaluate generalization via simulation likelihood.
5. **Compare models**: rank by held-out criteria (joint NLL, BIC, and diagnostic decompositions).

### Core scoring terminology
- Trial joint loss: `nll_joint = -log p(choice) + -log p(rt | choice)`
- Lower held-out `nll_joint` indicates better predictive fit.

## 4. Validity and Interpretation Notes

- `all_blocks_nominal_before` indicates whether each participant-block had the full expected number of trials before preprocessing/exclusions.
- Train/test hygiene is preserved because active `H` is fit on TRAIN only, then reused for TEST scoring.
- Model validity checks should include:
  - held-out ranking (joint NLL and BIC),
  - choice vs RT conditional loss decomposition,
  - trial-level NLL distribution and participant-level consistency.

This is the intended interpretation layer for outputs produced in `workflow.ipynb`.